# Synthetic Data Generator Pipeline
### Step 3 : Synthetic Observations Timestamping Stage

## Overview

This notebook supports the synthetic pipeline stage documented by this notebook. It is part of the Synthetic support portion of the capstone workflow and should be read as a notebook-first implementation step rather than a separate pipeline redesign.


## Objectives

- Document how this notebook supports the synthetic pipeline stage documented by this notebook.
- Preserve the existing notebook-first execution flow, configuration usage, logger behavior, ledger behavior, and artifact outputs.
- Make the notebook easier to review by separating setup, validation, processing, outputs, and handoff context.
- Clarify whether the notebook is a support, testing, streaming, or Bronze-handoff step in the synthetic path.


## Code Reference

A detailed code-cell reference for this notebook is maintained in the project documentation:`docs/technical_reference/notebook_code_references/synthetic_03_sythetic_observations_timestamped_stage_code_reference.md`


In [ ]:
import os 

from utils.core.env_helpers import (
    env_float, 
    env_int, 
    env_str,
)

from utils.database.postgres import (
    get_engine_from_env, 
    read_sql_dataframe,
)

from utils.synthetic.pipeline.timestamp_stage_writer import (
    ensure_simulation_timing_config_table,
    insert_simulation_timing_config,
    build_observations_timestamped_stage,
    validate_observations_timestamped_stage,
)


In [ ]:
SCHEMA = env_str("CAPSTONE_SCHEMA", "capstone")

DATASET_ID = env_str(
    "SYNTHETIC_DATASET_ID",
    "pump_synthetic_v1",
    aliases=("DATASET_ID",),
)

RUN_ID = env_str(
    "SYNTHETIC_RUN_ID",
    "synthetic_run_001",
    aliases=("RUN_ID",),
)

IF_EXISTS_FLAG = "replace"
CHUNK_SIZE = env_int("SYNTHETIC_TIMESTAMP_SOURCE_ROW_CHUNK_SIZE", 250000)

SIMULATION_TIME_CONFIG_TABLE_NAME = env_str(
    "SYNTHETIC_SIMULATION_TIMING_CONFIG_TABLE",
    "simulation_timing_config",
)

SIMULATION_START_DATETIME = env_str(
    "SYNTHETIC_SIMULATION_START_DATETIME",
    "2026-04-06 12:30:00+00:00",
)

SIMULATION_SAMPLING_INTERVAL_SECONDS = env_float(
    "SYNTHETIC_SIMULATION_SAMPLING_INTERVAL_SECONDS",
    60.0,
)

TIMESTAMPED_SOURCE_TABLE_NAME = env_str(
    "SYNTHETIC_PREMELT_OBSERVATIONS_TABLE",
    "synthetic_observations_premelt_stage",
)

TIMESTAMPED_DESTINATION_TABLE_NAME = env_str(
    "SYNTHETIC_TIMESTAMPED_OBSERVATIONS_TABLE",
    "synthetic_observations_timestamped_stage",
)

print("Timestamp config")
print("schema:", SCHEMA)
print("dataset id:", DATASET_ID)
print("run id:", RUN_ID)
print("source table:", TIMESTAMPED_SOURCE_TABLE_NAME)
print("target table:", TIMESTAMPED_DESTINATION_TABLE_NAME)
print("timing config table:", SIMULATION_TIME_CONFIG_TABLE_NAME)
print("simulation start:", SIMULATION_START_DATETIME)
print("sampling interval seconds:", SIMULATION_SAMPLING_INTERVAL_SECONDS)

---

In [10]:
engine = get_engine_from_env()


---

In [11]:

ensure_simulation_timing_config_table(
    engine=engine,
    schema=SCHEMA,
    table_name=SIMULATION_TIME_CONFIG_TABLE_NAME,
)


'simulation_timing_config'

---

In [12]:

insert_simulation_timing_config(
    engine=engine,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    simulation_start_datetime=SIMULATION_START_DATETIME,
    sampling_interval_seconds=SIMULATION_SAMPLING_INTERVAL_SECONDS,
    schema=SCHEMA,
    table_name=SIMULATION_TIME_CONFIG_TABLE_NAME,
    set_active=True,
    deactivate_existing_for_run=True,
)

print("Timing config ready.")

Timing config ready.


---

In [13]:
timestamped_table_name = build_observations_timestamped_stage(
    engine=engine,
    schema=SCHEMA,
    source_table=TIMESTAMPED_SOURCE_TABLE_NAME,
    target_table=TIMESTAMPED_DESTINATION_TABLE_NAME,
    timing_config_table=SIMULATION_TIME_CONFIG_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    if_exists=IF_EXISTS_FLAG,
    chunk_size=CHUNK_SIZE,
)

print("Built table:", timestamped_table_name)


[chunk] 1 | source rows 0 to 71,999
[timestamp] wrote chunk 1 with 72,000 rows
Built table: synthetic_observations_timestamped_stage


---

In [14]:
validation_dataframe = validate_observations_timestamped_stage(
    engine=engine,
    schema=SCHEMA,
    table_name=TIMESTAMPED_DESTINATION_TABLE_NAME,
)

display(validation_dataframe)

,row_count,distinct_observation_count,min_observation_timestamp,max_observation_timestamp,distinct_observation_timestamp_count
0,72000,72000,2026-04-06 12:30:00+00:00,2026-05-26 12:29:00+00:00,72000


---

In [ ]:

sample_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        observation_index,
        observation_timestamp,
        generated_row_id,
        stream_state,
        phase,
        meta_episode_id,
        sensor_00,
        sensor_01
    FROM "{SCHEMA}"."{TIMESTAMPED_DESTINATION_TABLE_NAME}"
    ORDER BY observation_index
    LIMIT 10
    """
)

display(sample_dataframe)


,observation_index,observation_timestamp,generated_row_id,stream_state,phase,meta_episode_id,sensor_00,sensor_01
0,1,2026-04-06 12:30:00+00:00,premelt_run_001_obs_000000000001,normal,normal,0,2.509287,49.445523
1,2,2026-04-06 12:31:00+00:00,premelt_run_001_obs_000000000002,normal,normal,0,2.472567,46.189799
2,3,2026-04-06 12:32:00+00:00,premelt_run_001_obs_000000000003,normal,normal,0,2.461393,46.169400
3,4,2026-04-06 12:33:00+00:00,premelt_run_001_obs_000000000004,normal,normal,0,2.517065,47.518164
4,5,2026-04-06 12:34:00+00:00,premelt_run_001_obs_000000000005,normal,normal,0,2.211810,44.307424
5,6,2026-04-06 12:35:00+00:00,premelt_run_001_obs_000000000006,normal,normal,0,1.994832,46.181536
6,7,2026-04-06 12:36:00+00:00,premelt_run_001_obs_000000000007,normal,normal,0,2.236349,47.763739
7,8,2026-04-06 12:37:00+00:00,premelt_run_001_obs_000000000008,normal,normal,0,2.360409,44.688221
8,9,2026-04-06 12:38:00+00:00,premelt_run_001_obs_000000000009,normal,normal,0,2.413066,45.612344
9,10,2026-04-06 12:39:00+00:00,premelt_run_001_obs_000000000010,normal,normal,0,2.207526,49.515945


---


In [16]:
timestamp_check_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        observation_index,
        observation_timestamp,
        stream_state,
        phase,
        meta_episode_id
    FROM capstone.synthetic_observations_timestamped_stage
    ORDER BY observation_index
    LIMIT 10
    """
)

display(timestamp_check_dataframe)


,observation_index,observation_timestamp,stream_state,phase,meta_episode_id
0,1,2026-04-06 12:30:00+00:00,normal,normal,0
1,2,2026-04-06 12:31:00+00:00,normal,normal,0
2,3,2026-04-06 12:32:00+00:00,normal,normal,0
3,4,2026-04-06 12:33:00+00:00,normal,normal,0
4,5,2026-04-06 12:34:00+00:00,normal,normal,0
5,6,2026-04-06 12:35:00+00:00,normal,normal,0
6,7,2026-04-06 12:36:00+00:00,normal,normal,0
7,8,2026-04-06 12:37:00+00:00,normal,normal,0
8,9,2026-04-06 12:38:00+00:00,normal,normal,0
9,10,2026-04-06 12:39:00+00:00,normal,normal,0


---

In [17]:
# Dispose Engine
engine.dispose()

## Summary

This notebook preserves the current analytical behavior while documenting the role of the Synthetic support step in the capstone workflow. The code cells above should continue to produce the same configured outputs, artifacts, logger messages, and ledger entries as before this documentation pass.


## Next Stage

Timestamped observations feed sensor-message and send-queue preparation.
